# venv
py -m venv .venv
source .venv/bin/activate
deactivate

# pip
pip install openai
pip install torch pandas transformers
pip install requests
pip install ipykernel 

# ollama
ollama start
ollama run llama3.1:8b

In [6]:
# from openai import OpenAI
from openai import AsyncOpenAI

client = AsyncOpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama',  
)

model='llama3.1:8b'

In [7]:
import json
import requests

url = "http://localhost:11434/api/show"
payload = {"name": "llama3.1:8b"}

response = requests.post(url, json=payload)
if response.status_code == 200:
    config_data = response.json()
    # Красивый вывод JSON со всеми параметрами модели
    print(json.dumps(config_data, indent=2, ensure_ascii=False))
else:
    print(f"Ошибка запроса: {response.status_code}")


{
  "license": "LLAMA 3.1 COMMUNITY LICENSE AGREEMENT\nLlama 3.1 Version Release Date: July 23, 2024\n\n“Agreement” means the terms and conditions for use, reproduction, distribution and modification of the\nLlama Materials set forth herein.\n\n“Documentation” means the specifications, manuals and documentation accompanying Llama 3.1\ndistributed by Meta at https://llama.meta.com/doc/overview.\n\n“Licensee” or “you” means you, or your employer or any other person or entity (if you are entering into\nthis Agreement on such person or entity’s behalf), of the age required under applicable laws, rules or\nregulations to provide legal consent and that has legal authority to bind your employer or such other\nperson or entity if you are entering in this Agreement on their behalf.\n\n“Llama 3.1” means the foundational large language models and software and algorithms, including\nmachine-learning model code, trained model weights, inference-enabling code, training-enabling code,\nfine-tuning en

In [19]:
import re

EXPECTED_FIELDS = {
    "product_id",
    "description",
    "pros",
    "cons",
    "tags",
}

MIN_DESCRIPTION_LENGTH = 120 
MAX_DESCRIPTION_LENGTH = 700 


class ResponseValidationError(ValueError):
    """Ошибка парсинга или проверки ответа модели."""


def count_sentences(text):
    """Число предложений в тексте. Точка внутри числа концом предложения не считается."""
    return len([s for s in re.split(r"[.!?]+(?!\d)", text) if s.strip()]) 


def parse_and_validate(content: str, product_id_req: str) -> dict[str, str]:
    """Преобразовать JSON-ответ и проверить его контракт."""
    # print('>>>>>\n', content)
    try:
        data = json.loads(content)
    except json.JSONDecodeError as error:
        raise ResponseValidationError(
            f"Ответ не является корректным JSON: {error.msg}"
        ) from error

    if not isinstance(data, dict):
        raise ResponseValidationError(
            "На верхнем уровне должен находиться JSON-объект"
        )

    received_fields = set(data)
    missing_fields = EXPECTED_FIELDS - received_fields
    unexpected_fields = received_fields - EXPECTED_FIELDS

    if missing_fields:
        raise ResponseValidationError(
            "Отсутствуют обязательные поля: "
            f"{sorted(missing_fields)}"
        )

    if unexpected_fields:
        raise ResponseValidationError(
            "Получены лишние поля: "
            f"{sorted(unexpected_fields)}"
        )

    product_id = data['product_id']
    if product_id != product_id_req:
        raise ResponseValidationError("Поле product_id не соответствует входным параметрам")

    description = data["description"]

    if not isinstance(description, str):
        raise ResponseValidationError("Поле description должно быть строкой")

    description = description.strip()

    if not description:
      raise ResponseValidationError("Поле description не должно быть пустым")
      
    i = count_sentences(description)
    if i < 2 or i > 5:
      raise ResponseValidationError("Description must contain 2-5 sentences")

    if len(description) < MIN_DESCRIPTION_LENGTH:
      raise ResponseValidationError(
          "Поле description содержит больше "
          f"{MAX_DESCRIPTION_LENGTH} символов"
      )

    if len(description) > MAX_DESCRIPTION_LENGTH:
      raise ResponseValidationError(
          "Поле description содержит меньше "
          f"{MAX_DESCRIPTION_LENGTH} символов"
      )

    pros = data['pros']
    if len(pros) < 2 or len(pros) > 5:
       raise ResponseValidationError("Поле pros must contain 2-5 values")

    cons = data['cons']
    if len(pros) < 1 or len(pros) > 3:
       raise ResponseValidationError("Поле cons must contain 1-3 values")
    
    tags = data['tags']
    if len(pros) < 3 or len(pros) > 8:
       raise ResponseValidationError("Поле cons must contain 3-8 values")
    
    return {
        "product_id": product_id,
        "description": description,
        "pros": pros,
        "cons": cons,
        "tags": tags,
    }

In [10]:
from typing import List
from typing_extensions import Annotated

from pydantic import (
    AfterValidator,
    BaseModel,
    ConfigDict,
    Field,
    field_validator,
)

class CardRequest(BaseModel):
    """Product Card"""

    model_config = ConfigDict(extra="forbid")

    product_id: str = Field(min_length=1, max_length=50)
    description: Annotated[str, Field(min_length=120, max_length=700), AfterValidator(count_sentences)]
    pros: Annotated[List[str], Field(min_length=2, max_length=5)]
    cons: Annotated[List[str], Field(min_length=1, max_length=3)]
    tags: Annotated[List[str], Field(min_length=3, max_length=8)]


    @field_validator("description")
    @classmethod
    def validate_description(cls, value: str) -> str:
      """Удалить внешние пробелы и проверить описание."""
      value = value.strip()

      if not value:
        raise ValueError(
            "Поле summary не должно быть пустым"
        )
      
      i = count_sentences(value)
      if i < 2 or i > 5:
        raise ValueError("Description must contain 2-5 sentences")
      
      return value


    @field_validator('pros')
    @classmethod
    def check_pros(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы pros не могут быть пустыми строками")
      return v


    @field_validator('cons')
    @classmethod
    def check_cons(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы cons не могут быть пустыми строками")
      return v


    @field_validator('tags')
    @classmethod
    def check_tags(cls, v: List[str]) -> List[str]:
      for i in v:
        if not i.strip():
          raise ValueError("Элементы tags не могут быть пустыми строками")
      return v

In [11]:
response_schema = CardRequest.model_json_schema()

In [12]:
response_schema

{'additionalProperties': False,
 'description': 'Product Card',
 'properties': {'product_id': {'maxLength': 50,
   'minLength': 1,
   'title': 'Product Id',
   'type': 'string'},
  'description': {'maxLength': 700,
   'minLength': 120,
   'title': 'Description',
   'type': 'string'},
  'pros': {'items': {'type': 'string'},
   'maxItems': 5,
   'minItems': 2,
   'title': 'Pros',
   'type': 'array'},
  'cons': {'items': {'type': 'string'},
   'maxItems': 3,
   'minItems': 1,
   'title': 'Cons',
   'type': 'array'},
  'tags': {'items': {'type': 'string'},
   'maxItems': 8,
   'minItems': 3,
   'title': 'Tags',
   'type': 'array'}},
 'required': ['product_id', 'description', 'pros', 'cons', 'tags'],
 'title': 'CardRequest',
 'type': 'object'}

In [13]:
async def request_one(
    msg: dict, 
    params={"temperature": 0.3, "top_p": 0.9, "top_k": 50, "repetition_penalty": 1.1, "max_tokens": 350}
) -> str:
  response = await client.chat.completions.create(
      model=model,  
      messages=[
          {"role": "system", "content": 
            """
              Generate a product card in Russian (utf-8) by example.
              Example:
              {
                "product_id": "P-1001",
                "description": "Компактные внутриканальные наушники с Bluetooth 5.3 и защитой IPX4. Держат заряд до 24 часов и надёжно сидят при активном движении. Подойдут для спорта и повседневного использования.",
                "pros": ["Чистый звук", "Удобная посадка", "Долгое время работы"],
                "cons": ["Быстро разряжаются на морозе"],
                "tags": ["наушники", "беспроводные", "спорт", "IPX4", "Bluetooth 5.3"]
              } 
              "description": 2-5 sentences
            """
          },
          {"role": "user", "content": f"""{msg}"""},
      ],
      temperature=params['temperature'],
      top_p=params['top_p'],
      max_tokens=params['max_tokens'],
      response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "support_request",
            "strict": True,
            "schema": CardRequest.model_json_schema(),
        },
      },
      extra_body={
        "provider": {
          "require_parameters": True,
          "top_k": params['top_k'], 
          "repetition_penalty": params['repetition_penalty'],
        },
      },
  )

  content = response.choices[0].message.content
  if not content:
    raise ResponseValidationError(
        "Empty response of model"
    )
  
  return content

In [36]:
import json

MAX_ATTEMPTS = 3 

async def generate_one(
    msg: str, 
    params={"temperature": 0.3, "top_p": 0.9, "top_k": 50, "repetition_penalty": 1.1, "max_tokens": 350}
) -> str:
  messages_history = json.loads(msg).copy()
  # messages_history = msg.copy()

  for attempt in range(1, MAX_ATTEMPTS + 1):
    content = await request_one(messages_history, params)
    return content

    try:
      parse_and_validate(content, msg['product_id'])
      return content

    except ResponseValidationError as error:
      if attempt == MAX_ATTEMPTS:
        return {"error": "ResponseValidationError"}

      messages_history.extend(
        [
          {
            "role": "assistant",
            "content": content,
          },
          {
            "role": "user",
            "content": (
                "Предыдущий ответ не прошёл проверку. "
                f"Причина: {error}. "
                "Исправь ответ и верни только JSON"
            ),
          },
        ]
      ) 

In [26]:
import asyncio, time

async def run_concurrent(items, concurrency, generate_one, params):
  """Отправляет запросы параллельно, не больше `concurrency` одновременно.

  generate_one — ваша асинхронная функция, принимает один товар и возвращает
  словарь с результатом. Возвращает список результатов и общее время прогона.
  """
  sem = asyncio.Semaphore(concurrency)

  async def wrapped(item):
    async with sem:
      return await generate_one(item, params)

  started = time.perf_counter()
  results = await asyncio.gather(*[wrapped(it) for it in items])
  return results, time.perf_counter() - started

In [16]:
file_name = './dev.jsonl'
with open(file_name, 'r', encoding='utf-8') as file:
  items = file.read().splitlines()

In [22]:
import json
from datetime import datetime
from pathlib import Path

def wright_files(timestamp_str_dir, results, params, df):
  timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
  directory = Path(f"./outputs/{timestamp_str_dir}/{timestamp_str}")
  directory.mkdir(parents=True, exist_ok=True)

  with open(directory / "results.json", "w", encoding="utf-8") as file:
    json.dump(results, file, indent=4, ensure_ascii=False)

  with open(directory / "params.json", "w", encoding="utf-8") as file:
    json.dump(params, file, indent=4, ensure_ascii=False)

  totals_path = f"./outputs/{timestamp_str_dir}" / "totals.csv"
  df.to_csv(totals_path, index=False, encoding="utf-8")

In [ ]:
import pandas as pd

its = items[:3]

df = pd.DataFrame(columns=['concurrency', "temperature", "top_p", "top_k", "repetition_penalty", "max_tokens", 'total_time', 'time_per_item'])
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")

# for concurrency in [1, 2, 4]:
#   for temperature in [0.0, 0.3, 0.7]:
#     for top_p in [0.9, 1.0]:
#       for top_k in [20, 50, None]:
#         for repetition_penalty in [1.0, 1.1, 1.2]:
#           for max_tokens in [250, 350, 500]:
for concurrency in [1]:
  for temperature in [0.0]:
    for top_p in [0.9]:
      for top_k in [20]:
        for repetition_penalty in [1.0]:
          for max_tokens in [250]:
            params = {
              "temperature": temperature, 
              "top_p": top_p, 
              "top_k": top_k, 
              "repetition_penalty": repetition_penalty, 
              "max_tokens": max_tokens,
            }

            results, time_total = await run_concurrent(its, concurrency, generate_one, params)

            new_row = {'concurrency': concurrency} | params | {'total_time': time_total, 'time_per_item': round(time_total/len(its), 2)}
            print(new_row)

            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)

            wright_files(timestamp_str, results, new_row, df)

In [ ]:
df_sorted = df.sort_values(by='time_per_item')

totals_path = Path(f"./outputs/{timestamp_str}") / "totals_final.csv"
df_sorted.to_csv(totals_path, index=False, encoding="utf-8")

print(df_sorted)